# Task 3: Prediction of perturbation effect

In [1]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from scmlcourse.perturbation import *

/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [2]:
data_dir = Path("../../data")

### Data Loading


In [3]:
data_module = DataModule(data_dir/"qced_data.h5ad")

/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:696: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["hvg"] = {"flavor": flavor}
/home/ntbiotech/Nextcloud/Documents/seminars/scMachineLearning/project/scMLCourse/src/scmlcourse/perturbation.py:53: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  self.adata.obs["perturbed_gene"] = self.adata.obs["perturbed_gene"].map(lambda x: "Ctrl" if pd.isna(x) else x)
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:431: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "

['SDC3' 'NT5E' 'CTSD' 'CDKN1A' 'WNT7A' 'JAK1' 'FGFR1' 'FSTL3' 'GSN' 'CD44'] Index(['NO', 'ONE', 'IFNGR2', 'JAK2', 'MFGE8', 'SLC22A18', 'NUP50-AS1', 'PSAP',
       'ACTA2', 'RTP4', 'CSPG4', 'HLA-A', 'TGFB1', 'STAT1', 'HLA-DRB5', 'B2M',
       'RB1', 'FMN1', 'TSC22D3', 'LCP1', 'CDH19', 'CD59', 'JMJD7', 'AEBP1',
       'IDH2', 'TTLL1', 'SLC5A3', 'SMAD3', 'TYR', 'A2M', 'C19orf48', 'CD274',
       'LEF1-AS1', 'FBXO32', 'NPC1', 'LINC00518', 'KDR', 'LAMP2', 'WBP2',
       'SOX4'],
      dtype='str', name='perturbed_gene')


## Simplistic Baseline: Small MLP

In [4]:
from pytorch_lightning.loggers import CSVLogger
log_dir = data_dir/"logs"
log_dir.mkdir(exist_ok=True)
logger = CSVLogger(log_dir)

In [5]:
mlp_config = dict(
    in_dim=data_module.n_vars,
    out_dim=data_module.n_vars,
    hidden_dims=(data_module.n_vars,)
)
model = Baseline(mlp_config)


In [6]:
train(model, data_module, max_epochs=1)

Seed set to 0
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:479: You passed `Trainer(accelerator='cpu', precision='16-mixed')` but AMP with fp16 is not supported on CPU. Using `precision='bf16-mixed'` instead.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name   | Type    | Params | Mode  | FLOPs
----------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:485: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0:   0%|          | 1/358 [00:00<03:16,  1.81it/s, v_num=3, train_loss_step=124.0]

...

Metric val_loss improved. New best score: 122.437
`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 358/358 [04:05<00:00,  1.46it/s, v_num=3, train_loss_step=125.0, val_loss=122.0, val_pearson=-0.0189, train_loss_epoch=122.0]

Restoring states from the checkpoint path at logs/pert_regressor/version_3/checkpoints/epoch=0-val_loss=122.4371.ckpt
Loaded model weights from the checkpoint at logs/pert_regressor/version_3/checkpoints/epoch=0-val_loss=122.4371.ckpt
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:485: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
/home/ntbiotech/.miniforge/envs/scml/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.



Testing DataLoader 0: 100%|██████████| 65/65 [00:41<00:00,  1.56it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            169.6796417236328
      test_pearson         -0.015018644742667675
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


(Baseline(
   (module): MLP(
     (net): Sequential(
       (0): Linear(in_features=1021, out_features=1021, bias=True)
       (1): ReLU()
       (2): Dropout(p=0.1, inplace=False)
       (3): Linear(in_features=1021, out_features=1021, bias=True)
     )
   )
   (loss): MSELoss()
 ),
 [{'test_loss': 169.6796417236328, 'test_pearson': -0.015018644742667675}])